# Lesson 02 ? Crafting an ML Pipeline with scikit-learn

We'll build a tidy machine learning workflow that predicts apartment prices. Along the way we'll combine feature engineering, preprocessing, and model training into a repeatable pipeline.

## Learning Goals
- Simulate a structured dataset with both numeric and categorical features
- Assemble preprocessing steps with a `ColumnTransformer`
- Train, evaluate, and tune a `RandomForestRegressor`
- Package the whole workflow into a scikit-learn `Pipeline`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error

np.random.seed(7)

## Step 1 ? Generate a Realistic Dataset
To keep things self-contained we'll synthesize a DataFrame that mimics apartment listings in a fictional city.

In [ ]:
def generate_apartment_data(n_samples=800):
    neighborhoods = np.random.choice([
        "Downtown", "Midtown", "Riverside", "Uptown"
    ], size=n_samples, p=[0.35, 0.25, 0.2, 0.2])

    base_price = {
        "Downtown": 3200,
        "Midtown": 2600,
        "Riverside": 2200,
        "Uptown": 2400,
    }

    sqft = np.random.normal(750, 180, size=n_samples).clip(350, 1400)
    bedrooms = np.random.choice([1, 2, 3], size=n_samples, p=[0.5, 0.35, 0.15])
    has_balcony = np.random.choice([0, 1], size=n_samples, p=[0.7, 0.3])
    floor = np.random.randint(1, 40, size=n_samples)
    walk_score = np.random.normal(80, 12, size=n_samples).clip(40, 100)

    prices = np.array([base_price[n] for n in neighborhoods])
    prices += (sqft - 700) * 2.5
    prices += bedrooms * 200
    prices += has_balcony * 150
    prices += (floor > 20) * 120
    prices += (walk_score - 70) * 8
    prices += np.random.normal(0, 180, size=n_samples)

    df = pd.DataFrame({
        "neighborhood": neighborhoods,
        "sqft": sqft.astype(int),
        "bedrooms": bedrooms,
        "has_balcony": has_balcony,
        "floor": floor,
        "walk_score": walk_score.round(1),
        "monthly_rent": prices.round(0)
    })
    return df


data = generate_apartment_data()
data.head()

### Train/Test Split
We'll hold out 20% of the data to estimate generalization performance.

In [ ]:
target = "monthly_rent"
features = data.drop(columns=[target])
labels = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

## Step 2 ? Assemble the Pipeline
We'll standardize numeric columns, one-hot encode categoricals, and feed everything to a Random Forest.

In [ ]:
numeric_features = ["sqft", "bedrooms", "floor", "walk_score"]
categorical_features = ["neighborhood", "has_balcony"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=3,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", model)
])

pipeline

### Train the Pipeline
All preprocessing happens inside `.fit()`?no data leakage, no manual juggling.

In [ ]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: ${mae:,.2f}")

### Cross-Validation Snapshot
A quick cross-validation run shows how stable the pipeline is across different data folds.

In [ ]:
cv_scores = cross_val_score(pipeline, features, labels, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1)
print("Cross-validated MAE (lower is better):")
for fold, score in enumerate(-cv_scores, start=1):
    print(f"  Fold {fold}: ${score:,.2f}")
print(f"Average: ${(-cv_scores).mean():,.2f}")

### Peek Inside the Model
Random forests provide feature importances. We have to recover them from the pipeline by matching columns after one-hot encoding.

In [ ]:
def pipeline_feature_importances(pipeline):
    preprocess = pipeline.named_steps["preprocess"]
    model = pipeline.named_steps["model"]

    # Numeric features stay as-is after StandardScaler
    num_features = preprocess.transformers_[0][2]

    # OneHotEncoder expands categorical features
    ohe = preprocess.transformers_[1][1]
    cat_features = preprocess.transformers_[1][2]
    cat_feature_names = list(ohe.get_feature_names_out(cat_features))

    feature_names = list(num_features) + cat_feature_names
    importances = model.feature_importances_
    return pd.Series(importances, index=feature_names).sort_values(ascending=False)


feature_importances = pipeline_feature_importances(pipeline)
feature_importances.head(10)

## Wrap-Up
You now have a full-stack structured data pipeline: data prep, model training, evaluation, and introspection all in one place. Try swapping the estimator, tweaking transformers, or exporting the pipeline with `joblib` to deploy it.